In [ ]:
import os
import json
import subprocess
import torch
from tqdm import tqdm

dataset = "cos_hsa_prostate_c"
SlideDir = f"PATH/TO/{dataset}/"
SlideOutputDir = f"PATH/TO/{dataset}/"
RunSummaryDir = "PATH/TO/RunSummary"
he_path = "PATH/TO/HE.tif"
tx_path = "PATH/TO/tx_file.csv.gz"

# Study-specific examples: replace these with the actual image metadata and regions.
COSMX_MPP = 0.12028
HE_MPP = 0.24891  # Micrometres per pixel of the study H&E image, not a CosMx default.
ALIGNMENT_FOV_RANGE = (1, 36)
ANALYSIS_FOV_RANGE = (90, 93)
DENSITY_BIN_SIZE = 8
CROP_BOUNDS = (150, 4351, 200, 4401)  # y_start, y_stop, x_start, x_stop; stop is exclusive.
STITCH_IMAGES = "stitch-images"
READ_TARGETS = "read-targets"

if COSMX_MPP <= 0 or HE_MPP <= 0 or DENSITY_BIN_SIZE <= 0:
    raise ValueError("Pixel sizes and density bin size must be positive")
for bounds in (ALIGNMENT_FOV_RANGE, ANALYSIS_FOV_RANGE):
    if len(bounds) != 2 or bounds[0] > bounds[1]:
        raise ValueError("Specify each inclusive FOV range as (first, last)")
os.makedirs(SlideOutputDir, exist_ok=True)


# 1. preprocessing images

In [ ]:
CellStatsDir = os.path.join(SlideDir, "CellStatsDir")
AnalysisDirParent = os.path.join(SlideDir, "AnalysisResults")
AnalysisDirSubBasename = sorted(i for i in os.listdir(AnalysisDirParent) if not i.startswith("."))
if len(AnalysisDirSubBasename) != 1:
    raise ValueError("Select the intended AnalysisResults directory explicitly")
AnalysisDir = os.path.join(AnalysisDirParent, AnalysisDirSubBasename[0])
subprocess.run([STITCH_IMAGES, "-i", CellStatsDir, "-f", RunSummaryDir,
                "-o", SlideOutputDir, "--umperpx", str(COSMX_MPP)], check=True)
subprocess.run([READ_TARGETS, AnalysisDir, "-o", SlideOutputDir], check=True)


# 2. generate tx density

In [ ]:
import numpy as np
import pandas as pd
import tifffile as tiff
from typing import Optional, Tuple

def compute_min_tile_origin_from_tx(
    tx_gz_path: str,
    fov_min: int = 1,
    fov_max: int = 36,
    chunksize: int = 2_000_000,
) -> Tuple[float, float]:
    fov_keep = set(range(fov_min, fov_max + 1))
    min_tile0_x = np.inf
    min_tile0_y = np.inf
    reader = pd.read_csv(tx_gz_path, compression="gzip", chunksize=chunksize)
    for chunk in reader:
        chunk = chunk[chunk["fov"].astype(int).isin(fov_keep)]
        if len(chunk) == 0:
            continue
        tile0_x = chunk["x_global_px"].to_numpy(float) - chunk["x_local_px"].to_numpy(float)
        tile0_y = chunk["y_global_px"].to_numpy(float) - chunk["y_local_px"].to_numpy(float)
        min_tile0_x = min(min_tile0_x, np.nanmin(tile0_x))
        min_tile0_y = min(min_tile0_y, np.nanmin(tile0_y))
    if not np.isfinite(min_tile0_x):
        raise RuntimeError("No transcripts within FOV.")
    return float(min_tile0_x), float(min_tile0_y)


def make_tx_density_image(
    tx_gz_path: str,
    min_tile0_x: float,
    min_tile0_y: float,
    fov_min: int = 1,
    fov_max: int = 36,
    chunksize: int = 2_000_000,
    bin_size_px: int = 8,
    max_points: Optional[int] = None
) -> np.ndarray:
    fov_keep = set(range(fov_min, fov_max + 1))
    max_x = 0.0
    max_y = 0.0
    total = 0
    reader = pd.read_csv(tx_gz_path, compression="gzip", chunksize=chunksize)
    for chunk in reader:
        chunk = chunk[chunk["fov"].astype(int).isin(fov_keep)]
        if len(chunk) == 0:
            continue
        x = chunk["x_global_px"].to_numpy(float) - min_tile0_x
        y = chunk["y_global_px"].to_numpy(float) - min_tile0_y
        max_x = max(max_x, np.nanmax(x))
        max_y = max(max_y, np.nanmax(y))
        total += len(chunk)
        if max_points is not None and total >= max_points:
            break
    W = int(np.ceil(max_x / bin_size_px)) + 1
    H = int(np.ceil(max_y / bin_size_px)) + 1
    dens = np.zeros((H, W), dtype=np.uint32)
    total = 0
    reader2 = pd.read_csv(tx_gz_path, compression="gzip", chunksize=chunksize)
    for chunk in reader2:
        chunk = chunk[chunk["fov"].astype(int).isin(fov_keep)]
        if len(chunk) == 0:
            continue
        x = chunk["x_global_px"].to_numpy(float) - min_tile0_x
        y = chunk["y_global_px"].to_numpy(float) - min_tile0_y
        xi = np.floor(x / bin_size_px).astype(np.int64)
        yi = np.floor(y / bin_size_px).astype(np.int64)
        m = (xi >= 0) & (yi >= 0) & (xi < W) & (yi < H)
        xi = xi[m]; yi = yi[m]
        np.add.at(dens, (yi, xi), 1)
        total += len(chunk)
        if max_points is not None and total >= max_points:
            break
    return dens


In [ ]:
min_tile0_x, min_tile0_y = compute_min_tile_origin_from_tx(tx_path, *ALIGNMENT_FOV_RANGE)
dens = make_tx_density_image(
    tx_gz_path=tx_path,
    min_tile0_x=min_tile0_x,
    min_tile0_y=min_tile0_y,
    fov_min=ALIGNMENT_FOV_RANGE[0],
    fov_max=ALIGNMENT_FOV_RANGE[1],
    bin_size_px=DENSITY_BIN_SIZE,
)
np.save(os.path.join(SlideOutputDir, "den.npy"), dens)


## Selected FOVs and interactive registration

The ranges above are examples of selecting a subset of FOVs, not fixed study-wide requirements. The alignment range builds the transcript-density reference. The analysis range selects the transcripts to export. These ranges may differ when both use the same global coordinate system and the registration remains valid over the selected analysis region. Verify alignment there before export.

Image stitching and target preparation were completed once above. `read-targets` prepares transcript targets for the viewer; the density and quantitative coordinate export below read the original transcript table.

Complete H&E alignment interactively, following the [manufacturer's tutorial](https://nanostring-biostats.github.io/CosMx-Analysis-Scratch-Space/posts/napari-alignment-tools/) as needed. Keep the density layer aligned with the CosMx reference while registering H&E. Export the numerical layer transforms below; a viewer screenshot is not a quantitative H&E input.


# 4.napari running (den+he+dapi)

In [ ]:
import numpy as np
import tifffile as tiff
from napari_cosmx.gemini import Gemini

def launch_napari_with_tx_density(
    slide_output_dir: str,
    he_path: str,
    tx_density_image: np.ndarray,
    cosmx_umperpx: float,
    he_umperpx: float,
    bin_size_px: int = 8,   # must same as make_tx_density_image
):
    gem = Gemini(slide_output_dir)
    if "Segmentation" in gem.viewer.layers:
        gem.viewer.layers["Segmentation"].visible = False
    if "FOV labels" in gem.viewer.layers:
        gem.viewer.layers["FOV labels"].visible = False
    gem.add_channel("DNA", colormap="gray")

    for layer in gem.viewer.layers:
        if getattr(layer, "_type_string", "") == "image":
            layer.scale = (cosmx_umperpx, cosmx_umperpx)
    # HE
    he = tiff.imread(he_path)
    he_layer = gem.viewer.add_image(
        he,
        name="HE",
        rgb=(he.ndim == 3 and he.shape[-1] in (3,4)),
        blending="additive",
        opacity=0.6,
        scale=(he_umperpx, he_umperpx),
    )

    # dens: scale = cosmx_umperpx * bin_size_px
    tx_layer = gem.viewer.add_image(
        np.log1p(tx_density_image).astype(np.float32),
        name="TX_density",
        blending="additive",
        opacity=0.45,
        scale=(cosmx_umperpx * bin_size_px, cosmx_umperpx * bin_size_px),
    )

    gem.viewer.layers.move(gem.viewer.layers.index(he_layer), 0)
    gem.show_widget()
    return gem


In [ ]:
gem = launch_napari_with_tx_density(
    slide_output_dir=SlideOutputDir,
    he_path=he_path,
    tx_density_image=dens,
    cosmx_umperpx=COSMX_MPP,
    he_umperpx=HE_MPP,
    bin_size_px=DENSITY_BIN_SIZE,
)


# 5.export and apply affine matrix

In [ ]:
import numpy as np

def _get_data2world_matrix(layer):
    """Recover the complete 2D affine transform, including rotation and shear."""
    if callable(getattr(layer, "data_to_world", None)):
        points = np.array([layer.data_to_world(p) for p in [(0, 0), (1, 0), (0, 1)]], dtype=float)
        if points.shape != (3, 2):
            raise ValueError("Expected a two-dimensional image layer")
        origin = points[0]
        matrix = np.eye(3)
        matrix[:2, :2] = (points[1:] - origin).T
        matrix[:2, 2] = origin
    else:
        try:
            matrix = np.asarray(layer._transforms["data2world"].affine_matrix, dtype=float)
        except (AttributeError, KeyError, TypeError) as error:
            raise RuntimeError("Cannot obtain the complete layer transform; use a compatible napari environment") from error
    if matrix.shape != (3, 3) or not np.isfinite(matrix).all() or abs(np.linalg.det(matrix)) < 1e-15:
        raise ValueError("Invalid or singular two-dimensional layer transform")
    return matrix

def export_T_density_px_to_he_px(gem, dens_layer_name="TX_density", he_layer_name="HE"):
    density = _get_data2world_matrix(gem.viewer.layers[dens_layer_name])
    histology = _get_data2world_matrix(gem.viewer.layers[he_layer_name])
    return np.linalg.solve(histology, density)


In [ ]:
T_dens_to_he = export_T_density_px_to_he_px(gem, "TX_density", "HE")
registration = {
    "dataset": dataset,
    "cosmx_mpp": COSMX_MPP,
    "he_mpp": HE_MPP,
    "alignment_fov_range": ALIGNMENT_FOV_RANGE,
    "analysis_fov_range": ANALYSIS_FOV_RANGE,
    "density_bin_size": DENSITY_BIN_SIZE,
    "mosaic_origin_xy": [min_tile0_x, min_tile0_y],
    "density_yx_to_he_yx": T_dens_to_he.tolist(),
    "crop_bounds_yxyx": CROP_BOUNDS,
}
with open(os.path.join(SlideOutputDir, "registration.json"), "w", encoding="utf-8") as handle:
    json.dump(registration, handle, indent=2)
print(T_dens_to_he)


In [ ]:
import numpy as np
import pandas as pd

def export_transcripts_use_from_density_transform(
    tx_gz_path,
    out_csv_path,
    T_dens_px_to_he_px,
    min_tile0_x,
    min_tile0_y,
    bin_size_px=8,
    fov_min=1,
    fov_max=36,
    chunksize=2_000_000,
    round_xy=None
):
    T = np.array(T_dens_px_to_he_px, dtype=float)
    if T.shape != (3,3):
        raise ValueError("T_dens_px_to_he_px must be 3x3")

    fov_keep = set(range(fov_min, fov_max + 1))
    out_chunks = []

    reader = pd.read_csv(tx_gz_path, compression="gzip", chunksize=chunksize)
    for chunk in reader:
        chunk = chunk[chunk["fov"].astype(int).isin(fov_keep)]
        if len(chunk) == 0:
            continue

        # tx -> DNA mosaic px
        x_dna = chunk["x_global_px"].to_numpy(float) - min_tile0_x
        y_dna = chunk["y_global_px"].to_numpy(float) - min_tile0_y

        # DNA mosaic px -> density px
        x_dens = x_dna / float(bin_size_px)
        y_dens = y_dna / float(bin_size_px)

        ones = np.ones_like(x_dens, dtype=float)

        pts = np.stack([y_dens, x_dens, ones], axis=0)  # 3xN
        out = T @ pts
        y_he = out[0, :]
        x_he = out[1, :]

        if round_xy is not None:
            x_he = np.round(x_he, round_xy)
            y_he = np.round(y_he, round_xy)

        out_chunks.append(pd.DataFrame({
            "gene": chunk["target"].astype(str).to_numpy(),
            "x": x_he,
            "y": y_he
        }))

    if not out_chunks:
        raise ValueError("No transcripts found in the selected analysis FOV range")
    out_df = pd.concat(out_chunks, ignore_index=True)
    out_df.to_csv(out_csv_path, index=False)
    return out_df


In [ ]:
df = export_transcripts_use_from_density_transform(
    tx_gz_path=tx_path,
    out_csv_path=os.path.join(SlideOutputDir, "transcripts-use-precrop.csv"),
    T_dens_px_to_he_px=T_dens_to_he,
    min_tile0_x=min_tile0_x,
    min_tile0_y=min_tile0_y,
    bin_size_px=DENSITY_BIN_SIZE,
    fov_min=ANALYSIS_FOV_RANGE[0],
    fov_max=ANALYSIS_FOV_RANGE[1],
    round_xy=None,
)
print(df.head())


In [ ]:
df.columns = ['feature_name','x_location','y_location']
df

In [ ]:
df.to_csv(f"{SlideOutputDir}transcripts-use-precrop.tsv", sep='\t', index=False)

# 6.check alignment

In [ ]:
import numpy as np
import pandas as pd
import tifffile as tiff
import matplotlib.pyplot as plt

def show_top_gene_overlay(
    he_path="he-raw.tiff",
    tx_path="transcripts-use.csv",
    max_points=200_000,      # max number of points to plot
    point_size=0.3, 
    alpha=0.25, 
    use_percentile_contrast=True,
    seed=0
):
    he = tiff.imread(he_path)
    tx = pd.read_csv(tx_path, sep=None, engine="python")
    tx = tx.rename(columns={"feature_name": "gene", "x_location": "x", "y_location": "y"})

    if not {"gene", "x", "y"}.issubset(tx.columns):
        raise ValueError("transcripts-use.csv must contain: gene, x, y")
    top_gene = tx["gene"].value_counts().idxmax()
    top_n = int(tx["gene"].value_counts().max())
    sub = tx[tx["gene"] == top_gene].copy()
    if len(sub) > max_points:
        sub = sub.sample(n=max_points, random_state=seed)
    he_show = he
    if he.ndim == 3 and he.shape[2] in (3, 4):
        pass
    else:
        if use_percentile_contrast:
            lo, hi = np.percentile(he, (1, 99))
            he_show = np.clip(he, lo, hi)
    plt.figure(figsize=(10, 10))
    if he_show.ndim == 3:
        plt.imshow(he_show)
    else:
        plt.imshow(he_show, cmap="gray")

    plt.scatter(sub["x"].to_numpy(), sub["y"].to_numpy(),
                s=point_size, alpha=alpha)

    plt.title(f"Overlay: top gene = {top_gene} (count={top_n}, shown={len(sub)})")
    plt.axis("off")
    plt.show()

    return top_gene, top_n

In [ ]:
top_gene, top_n = show_top_gene_overlay(
    he_path=he_path,
    tx_path=os.path.join(SlideOutputDir, "transcripts-use-precrop.tsv"),
    max_points=200_000,
    point_size=0.3,
    alpha=0.25,
)
print("Top gene:", top_gene, "count:", top_n)


# 7.cut he image

In [ ]:
import pandas as pd
transcripts_precrop = pd.read_csv(
    f"{SlideOutputDir}transcripts-use-precrop.tsv",
    sep='\t',
    header=0
)
transcripts_precrop

In [ ]:
print("min_x:" + str(transcripts_precrop['x_location'].min()))
print("max_x:" + str(transcripts_precrop['x_location'].max()))
print("min_y:" + str(transcripts_precrop['y_location'].min()))
print("max_y:" + str(transcripts_precrop['y_location'].max()))

In [ ]:
import tifffile
import matplotlib.pyplot as plt
he_precrop = tifffile.imread(he_path)
y_start, y_stop, x_start, x_stop = CROP_BOUNDS
if not (0 <= y_start < y_stop <= he_precrop.shape[0] and 0 <= x_start < x_stop <= he_precrop.shape[1]):
    raise ValueError("Crop bounds must lie inside the H&E image")
he = he_precrop[y_start:y_stop, x_start:x_stop]
plt.imshow(he)


In [ ]:
transcripts = transcripts_precrop.copy()
transcripts = transcripts[~transcripts["feature_name"].str.startswith("SystemControl", na=False)]
transcripts = transcripts[~transcripts["feature_name"].str.startswith("Negative", na=False)]
transcripts["feature_name"] = transcripts["feature_name"].str.replace("/", "_", regex=False)
transcripts['x_location'] = transcripts['x_location'] - x_start
transcripts['y_location'] = transcripts['y_location'] - y_start
xy = transcripts[["x_location", "y_location"]].to_numpy(dtype=float)
inside = (np.isfinite(xy).all(axis=1) & (xy[:, 0] >= 0) & (xy[:, 0] < he.shape[1])
          & (xy[:, 1] >= 0) & (xy[:, 1] < he.shape[0]))
if not inside.all():
    raise ValueError(f"{int((~inside).sum())} transcripts lie outside the crop; review the selected FOVs and crop bounds before continuing")
transcripts


In [ ]:
print("min_x:" + str(transcripts['x_location'].min()))
print("max_x:" + str(transcripts['x_location'].max()))
print("min_y:" + str(transcripts['y_location'].min()))
print("max_y:" + str(transcripts['y_location'].max()))

In [ ]:
len(set(transcripts['feature_name']))

In [ ]:
transcripts.to_csv(f"{SlideOutputDir}transcripts_use.tsv", sep='\t', index=False)
tifffile.imwrite(f"{SlideOutputDir}he-raw.tiff", he)
with open(os.path.join(SlideOutputDir, "pixel-size-raw.txt"), "w", encoding="utf-8") as handle:
    handle.write(str(HE_MPP))


In [ ]:
top_gene, top_n = show_top_gene_overlay(
    he_path=f"{SlideOutputDir}he-raw.tiff",
    tx_path=f"{SlideOutputDir}transcripts_use.tsv",
    max_points=400_000,
    point_size=0.3,
    alpha=0.5
)
print("Top gene:", top_gene, "count:", top_n)


# 8.Simulate Spot

In [ ]:
import pandas as pd
import seaborn as sns
import tifffile
from matplotlib import pyplot as plt
import numpy as np
import math

In [ ]:
from tqdm import tqdm
import shutil


In [ ]:
def gap_real_cal(resolution, simulation):
    if simulation == "proportional":
        return 100 / 55 * resolution
    elif simulation == "fixed_spotDistance":
        return 100
    elif simulation == "fixed_spotSize":
        return resolution
    elif simulation == "dense":
        return resolution
    else:
        raise ValueError("Illegal simulation type, please select one from [Proportional, Fixed_spotDistance, Fixed_spotSize, Dense]")

def spots_simulate_gpu(simulation, resolution, transcripts, output_dir, he_mpp):
    if not np.isfinite(he_mpp) or he_mpp <= 0:
        raise ValueError("Supply the positive physical pixel size of the H&E image")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    gap_real = gap_real_cal(resolution, simulation)
    if simulation == "fixed_spotSize":
        dir_prefix = "spotDistance_"
        spot_size_real = 55
        print(f"start gap: {resolution}")
    else:
        dir_prefix = "spotSize_"
        spot_size_real = resolution
        print(f"start spot_size: {resolution}")
        
    output_dir_full = f'{output_dir}/{simulation}/{dir_prefix}{resolution}/'
    os.makedirs(output_dir_full, exist_ok = True)
    
    gap = gap_real / he_mpp
    start_gap = gap / 2
    radius = spot_size_real / 2 / he_mpp
    
    # spot locations
    spots = pd.DataFrame(data=None, columns=['x','y'])
    min_x = transcripts['x_location'].min() + gap
    min_y = transcripts['y_location'].min() + gap
    max_x = transcripts['x_location'].max() - gap
    max_y = transcripts['y_location'].max() - gap
    
    xs = [ 
        np.arange(np.floor(min_x), max_x, gap), 
        np.arange(np.floor(min_x) + start_gap, max_x, gap) 
    ]
    ys = np.arange(np.floor(min_y), max_y, start_gap * math.sqrt(3))
    
    for i in range(len(ys)): 
        myrow = xs[i % 2]
        tmp = pd.DataFrame({'x': myrow, 'y': np.repeat(ys[i], len(myrow))})
        spots = pd.concat([spots, tmp], ignore_index=True)
        
    spots = spots.sort_values('x').reset_index(drop=True)
    spots.index = 'barcode' + spots.index.astype(str)
    spots['spot'] = spots.index
        
    spots.iloc[:,[2,0,1]].to_csv(f'{output_dir_full}/locs-raw.tsv', sep='\t', index=False)
    
    print(f"start cnts of {simulation} with {resolution}")
    
    spot_xy = torch.tensor(spots[['x', 'y']].to_numpy(dtype=np.float32), dtype=torch.float32, device=device)
    
    transcript_xy = torch.tensor(transcripts[['x_location', 'y_location']].to_numpy(dtype=np.float32), dtype=torch.float32, device=device)
    transcript_gene = transcripts['feature_name'].astype('category')
    gene_ids = torch.tensor(transcript_gene.cat.codes.values, device=device)
    all_gene_names = transcript_gene.cat.categories.tolist()
    cnts = torch.zeros((spot_xy.shape[0], len(all_gene_names)), dtype=torch.int32, device=device)
    for i in tqdm(range(spot_xy.shape[0])):
        center = spot_xy[i]  # shape [2]
        distances = torch.norm(transcript_xy - center, dim=1)  # [num_transcripts]
        mask = distances <= radius
        selected_genes = gene_ids[mask].long()  
        if selected_genes.numel() > 0:
            cnts[i].scatter_add_(0, selected_genes, torch.ones_like(selected_genes, dtype=torch.int32))
    cnts_df = pd.DataFrame(cnts.cpu().numpy(), index=spots.index, columns=all_gene_names)
    cnts_df.to_csv(f'{output_dir_full}/cnts.tsv', sep='\t', index=True, index_label='spot')
    
    print(f"finished {simulation} with {resolution}")


In [ ]:
# Default CosMx input: 55-um diameter, 100-um center spacing.
transcripts_use = pd.read_csv(os.path.join(SlideOutputDir, "transcripts_use.tsv"), sep='	')
default_matrix_root = os.path.join(SlideOutputDir, 'default_matrix')
spots_simulate_gpu('fixed_spotSize', 100, transcripts_use, default_matrix_root, HE_MPP)
default_matrix_dir = os.path.join(default_matrix_root, 'fixed_spotSize', 'spotDistance_100')
for filename in ['cnts.tsv', 'locs-raw.tsv']:
    shutil.copy2(os.path.join(default_matrix_dir, filename), os.path.join(SlideOutputDir, filename))
